In [18]:
import os
import re
from datetime import datetime

import ee
import geemap



#####################################
#    SET ALL PARAMETERS #############
#####################################

PROJECT_DIR = "projects/ee-tymc5571-goodfire/assets/"
GEE_PROJECT = "ee-tymc5571-goodfire"

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

ASSET_STRING = "FIREDv2"
SERVICE_FILE = "C:/Users/tymc5571/dev/cbi-module/config/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file
DIR_DERIVED = "C:/Users/tymc5571/dev/cbi-module/data/derived/"

In [4]:
def list_assets(parent: str) -> list[str]:
    """
    Recursively list asset IDs under an Earth Engine folder/collection.

    Args:
        parent: e.g. "projects/ee-tymc5571-goodfire/assets/"
    Returns:
        A list of child asset IDs under the given parent.
    Requires:
        import ee
        from typing import List
    """
    assets = []
    def _list(parent_id):
        response = ee.data.listAssets({'parent': parent_id})
        for item in response.get('assets', []):
            asset_id = item.get('name')
            asset_type = item.get('type')
            if asset_type in ('FOLDER', 'IMAGE_COLLECTION'):
                _list(asset_id)
            else:
                assets.append(asset_id)

    _list(parent)
    return assets

assets = list_assets(PROJECT_DIR)
filtered_assets = [a for a in assets if ASSET_STRING in a]
print(filtered_assets)

['projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2000_2004', 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2005_2009', 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2010_2014', 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2015_2019', 'projects/ee-tymc5571-goodfire/assets/FIREDv2_2025_cbi_bc_2020_2024']


In [4]:
images = [ee.Image(asset) for asset in filtered_assets]
image = images[0]
print('First image:', image.getInfo())

# Merge all bands
dats = images[1:]
for img in dats:
    image = image.addBands(img)

print('Merged image:', image.getInfo())

print(image.bandNames().getInfo())

# Load study areas
states = ee.FeatureCollection("TIGER/2018/States")

# Pacific Northwest
pnw = states.filter(ee.Filter.inList('NAME', ['Washington', 'Oregon']))
pnw_geom = pnw.geometry()

# # Export for PNW
# task1 = ee.batch.Export.image.toDrive(
#     image=image,
#     description='pnw_fired_bccbi',
#     folder='GEE_Exports',
#     fileNamePrefix='pnw_fired_bccbi',
#     region=pnw_geom,
#     scale=30,
#     crs='EPSG:4326',
#     maxPixels=1e13
# )
# task1.start()

# Extended Western US
west_states = [
    'Washington', 'Oregon', 'California', 'Nevada', 'Idaho', 'Montana',
    'Wyoming', 'Utah', 'Arizona', 'Colorado', 'New Mexico', 'Texas',
    'Oklahoma', 'Kansas', 'North Dakota', 'South Dakota', 'Nebraska'
]
west = states.filter(ee.Filter.inList('NAME', west_states))
west_geom = west.geometry()

# Export for extended western US
task2 = ee.batch.Export.image.toDrive(
    image=image,
    description='extendedwest_fired_bccbi',
    folder='GEE_Exports',
    fileNamePrefix='extendedwest_fired_bccbi',
    region=west_geom,
    scale=30,
    crs='EPSG:5070',
    maxPixels=1e13
)
task2.start()

First image: {'type': 'Image', 'bands': [{'id': 'year_2000', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [116296, 85959], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647, 0, -124.84902599680046, 0, -0.00026949458523585647, 49.00246992802102]}, {'id': 'year_2001', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [116296, 85959], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647, 0, -124.84902599680046, 0, -0.00026949458523585647, 49.00246992802102]}, {'id': 'year_2002', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [116296, 85959], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647, 0, -124.84902599680046, 0, -0.00026949458523585647, 49.00246992802102]}, {'id': 'year_2003', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'dimensions': [116296, 85959], 'crs': 'EPSG:4326', 'crs_transform': [0.00026949458523585647, 0, -124.84902599680046, 0, -0.00026949458523585647, 49.0

In [14]:
import os
import shutil
import tempfile
import time
import random
import contextlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Union, List
from pathlib import Path


import ee
import rioxarray as rxr
from rioxarray.merge import merge_arrays

import xarray as xr
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload



def list_assets(parent: str) -> list[str]:
    """
    Recursively list asset IDs under an Earth Engine folder/collection.

    Args:
        parent: e.g. "projects/ee-tymc5571-goodfire/assets/"
    Returns:
        A list of child asset IDs under the given parent.
    Requires:
        import ee
        from typing import List
    """
    assets = []
    def _list(parent_id):
        response = ee.data.listAssets({'parent': parent_id})
        for item in response.get('assets', []):
            asset_id = item.get('name')
            asset_type = item.get('type')
            if asset_type in ('FOLDER', 'IMAGE_COLLECTION'):
                _list(asset_id)
            else:
                assets.append(asset_id)

    _list(parent)
    return assets


def _download_file_from_drive(file_id, file_name, temp_dir, service_account_file):
    try:
        # Authenticate service account
        SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
        credentials = service_account.Credentials.from_service_account_file(
            os.path.abspath(service_account_file), scopes=SCOPES
        )
        service = build('drive', 'v3', credentials=credentials)

        request = service.files().get_media(fileId=file_id)
        local_path = os.path.join(temp_dir, file_name)
        with open(local_path, 'wb') as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
        time.sleep(random.uniform(0.5, 1.5))  # Random delay
        print(f"Downloaded {file_name}")
        return local_path
    except Exception as e:
        print(f"Error downloading {file_name}: {e}")
        return None


def download_merge_from_drive(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    compress: str = "deflate",
    check_existing: bool = True,
    n_workers: int = 1
) -> str:
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        print(f"File already exists: {local_filename}")
        return local_filename

    temp_dir = tempfile.mkdtemp()
    datasets = []

    try:
        SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
        credentials = service_account.Credentials.from_service_account_file(
            os.path.abspath(service_account_file), scopes=SCOPES
        )
        service = build('drive', 'v3', credentials=credentials)

        folder_results = service.files().list(
            q=f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
            fields="files(id, name)"
        ).execute()
        folders = folder_results.get('files', [])
        if not folders:
            raise FileNotFoundError(f"Folder '{drive_folder}' not found or not shared with service account.")
        folder_id = folders[0]['id']

        file_results = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false and name contains '{description}' and name contains '.tif'",
            fields="files(id, name)"
        ).execute()
        files = file_results.get('files', [])
        if not files:
            raise FileNotFoundError(f"No matching .tif files found for '{description}' in '{drive_folder}'.")

        print(f"Found {len(files)} files. Starting download...")

        downloaded_paths = []
        if n_workers == 1:
            for file in files:
                path = _download_file_from_drive(file['id'], file['name'], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as executor:
                futures = [
                    executor.submit(_download_file_from_drive, file['id'], file['name'], temp_dir, service_account_file)
                    for file in files
                ]
                for future in as_completed(futures):
                    result = future.result()
                    if result:
                        downloaded_paths.append(result)

        for f in downloaded_paths:
            ds = rxr.open_rasterio(f, masked=True, chunks=True)
            datasets.append(ds)

        mosaic = merge_arrays(datasets)
        mosaic.rio.to_raster(local_filename, compress=compress)
        print(f"Final merged GeoTIFF saved to: {local_filename}")
        return local_filename

    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise

    finally:
        for ds in datasets:
            with contextlib.suppress(Exception):
                ds.close()
        try:
            shutil.rmtree(temp_dir)
            print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")

def export_image_to_drive_and_download(
    image: ee.Image,
    region: ee.Geometry,
    description: str,
    local_filename: str,
    drive_folder: str = "EarthEngineExports",
    service_account_file: str = "your-service-account.json",
    scale: int = 30,
    wait_interval: int = 30,
    compress: str = "deflate",
    check_existing: bool = True,
    n_workers: int = 1
) -> str:
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        print(f"File already exists: {local_filename}")
        return local_filename

    task = ee.batch.Export.image.toDrive(
        image=image.clip(region),
        description=description,
        folder=drive_folder,
        fileNamePrefix=description,
        region=region.bounds().getInfo()["coordinates"],
        scale=scale,
        maxPixels=1e13
    )
    task.start()
    print(f"Started Earth Engine export: {description}")

    while task.active():
        print("Waiting for Earth Engine export to finish...")
        time.sleep(wait_interval)

    status = task.status()
    if status["state"] != "COMPLETED":
        raise RuntimeError(f"Export failed: {status}")

    print("Earth Engine export complete. Downloading from Drive...")

    return download_merge_from_drive(
        description=description,
        local_filename=local_filename,
        drive_folder=drive_folder,
        service_account_file=service_account_file,
        compress=compress,
        check_existing=check_existing,
        n_workers=n_workers
    )

In [ ]:
#30 minutes
download_merge_from_drive(
    description="extendedwest_fired_bccbi",
    local_filename=os.path.join(DIR_DERIVED, "extendedwest_fired_bccbi.tif"),
    drive_folder="GEE_Exports",
    service_account_file=SERVICE_FILE,
    compress="DEFLATE",         # same as 'deflate', case-insensitive
    predictor=3,                # 3 = best for float32 data (like indices or continuous values)
    blocksize=512,              # tile size; 256–512 is good default
    bigtiff="YES",              # ensures large mosaics don’t overflow 4 GB
    #dst_srs="EPSG:5070",        # optional: force reprojection to your CONUS Albers grid
    #xres=30, yres=30,           # optional: force 30 m pixels
    #dst_nodata=-9999,           # optional: set output NoData if needed
    check_existing=True,
    n_workers=4,                # parallel downloads from Drive
    include_all_drives=True,    # safer if GEE_Exports is on a Shared Drive
    verbose=True
)

Found 100 file(s). Starting download to C:\Users\tymc5571\AppData\Local\Temp\tmpl12wt497 ...
Downloaded extendedwest_fired_bccbi-0000093184-0000086528.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000079872.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000073216.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000066560.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000059904.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000046592.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000039936.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000053248.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000033280.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000026624.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000013312.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000019968.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000006656.tif
Downloaded extendedwest_fired_bccbi-0000093184-0000000000.tif
Downloaded extendedwest_fired_bccbi-000

c:\Users\tymc5571\AppData\Local\miniconda3\envs\cbi-module\lib\site-packages\osgeo\gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Translating VRT to tiled, compressed GeoTIFF (streamed)...
Final mosaicked GeoTIFF saved to: C:\Users\tymc5571\dev\cbi-module\data\derived\extendedwest_fired_bccbi.tif
Cleaned up temporary directory: C:\Users\tymc5571\AppData\Local\Temp\tmpl12wt497


'C:\\Users\\tymc5571\\dev\\cbi-module\\data\\derived\\extendedwest_fired_bccbi.tif'

In [16]:
import os
import shutil
import tempfile
import contextlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List

from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# GDAL bindings
from osgeo import gdal


def download_merge_from_drive(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    *,
    # Output & mosaic controls
    compress: str = "DEFLATE",          # DEFLATE|LZW|ZSTD|JPEG
    predictor: Optional[int] = None,     # 2 for integers, 3 for floats; None = let GDAL decide
    blocksize: int = 512,                # tile size for output GeoTIFF
    bigtiff: str = "YES",                # YES|IF_SAFER
    dst_srs: Optional[str] = None,       # e.g., "EPSG:5070" (if reprojection needed)
    xres: Optional[float] = None,        # set both xres & yres to enforce pixel size (e.g., 30)
    yres: Optional[float] = None,
    dst_nodata: Optional[float] = None,  # set if you need a specific nodata in output
    # Drive / listing controls
    drive_id: Optional[str] = None,      # if folder is in a Shared Drive, you can pass the Drive ID
    include_all_drives: bool = True,     # search across Shared Drives
    # Behavior controls
    check_existing: bool = True,
    n_workers: int = 4,
    verbose: bool = True,
) -> str:
    """
    Download all Drive .tif files whose names contain `description` from `drive_folder`,
    then build a VRT and stream a mosaicked GeoTIFF to `local_filename` (no in-memory merge).

    This avoids the huge memory allocations from rioxarray/rasterio.merge.
    """

    # --- Fast exit if we already have the target ---
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        if verbose:
            print(f"File already exists: {local_filename}")
        return local_filename

    # --- Temp workspace ---
    temp_dir = tempfile.mkdtemp()
    vrt_path = os.path.join(temp_dir, "mosaic.vrt")
    downloaded_paths: List[str] = []

    # --- Build Drive client ---
    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
    cred_path = os.path.abspath(service_account_file)
    credentials = service_account.Credentials.from_service_account_file(cred_path, scopes=SCOPES)
    service = build("drive", "v3", credentials=credentials)

    # Convenience flags for Shared Drives
    list_kwargs = dict(
        fields="files(id, name, parents)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ) if include_all_drives else dict(fields="files(id, name, parents)")

    try:
        # --- Locate the folder ---
        # If you know the Shared Drive ID, use corpora='drive' for speed; else search all.
        folder_list = service.files().list(
            q=f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        ).execute()

        folders = folder_list.get("files", [])
        if not folders:
            raise FileNotFoundError(
                f"Folder '{drive_folder}' not found. "
                "Ensure it exists and is shared with the service account "
                f"({credentials.service_account_email}). "
                "If it is on a Shared Drive, also ensure the service account has at least Viewer."
            )
        folder_id = folders[0]["id"]

        # --- List matching .tif files in that folder ---
        file_list = service.files().list(
            q=f"'{folder_id}' in parents and trashed=false and name contains '{description}' and name contains '.tif'",
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        ).execute()

        files = file_list.get("files", [])
        if not files:
            raise FileNotFoundError(
                f"No matching .tif files found for '{description}' in '{drive_folder}'. "
                "Check filenames and sharing."
            )

        if verbose:
            print(f"Found {len(files)} file(s). Starting download to {temp_dir} ...")

        # --- Download the files (parallel if requested) ---
        if n_workers <= 1:
            for f in files:
                path = _download_file_from_drive(f["id"], f["name"], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = [
                    ex.submit(_download_file_from_drive, f["id"], f["name"], temp_dir, service_account_file)
                    for f in files
                ]
                for fut in as_completed(futures):
                    p = fut.result()
                    if p:
                        downloaded_paths.append(p)

        if not downloaded_paths:
            raise RuntimeError("Downloads finished but no files were saved.")

        if verbose:
            print(f"Building VRT from {len(downloaded_paths)} tile(s) ...")

        # --- Build VRT (lightweight, no data copy) ---
        # Use srcNodata/VRTNodata if you need masking carried through the VRT.
        vrt_options = {}
        if dst_nodata is not None:
            vrt_options.update(dict(srcNodata=dst_nodata, VRTNodata=dst_nodata))

        vrt = gdal.BuildVRT(vrt_path, downloaded_paths, **vrt_options)
        if vrt is None:
            raise RuntimeError("GDAL BuildVRT failed.")
        vrt = None  # flush

        # --- Stream out to GeoTIFF (no giant array in RAM) ---
        co = [
            f"BIGTIFF={bigtiff}",
            "TILED=YES",
            f"COMPRESS={compress.upper()}",
            f"BLOCKXSIZE={int(blocksize)}",
            f"BLOCKYSIZE={int(blocksize)}",
        ]
        # PREDICTOR: 2 for ints, 3 for floats. If user passed a value, add it.
        if predictor is not None:
            co.append(f"PREDICTOR={int(predictor)}")

        # Decide between Translate (same grid) vs Warp (reproject/resample)
        needs_warp = any(v is not None for v in (dst_srs, xres, yres, dst_nodata))
        if needs_warp:
            if verbose:
                print("Warping (streamed) to target grid...")
            warp_kwargs = dict(
                format="GTiff",
                creationOptions=co,
                multithread=True,  # enables multi-threaded resampling where possible
            )
            if dst_srs:
                warp_kwargs["dstSRS"] = dst_srs
            if xres and yres:
                warp_kwargs["xRes"] = float(xres)
                warp_kwargs["yRes"] = float(yres)
                warp_kwargs["targetAlignedPixels"] = True
            if dst_nodata is not None:
                warp_kwargs["dstNodata"] = dst_nodata

            out_ds = gdal.Warp(local_filename, vrt_path, **warp_kwargs)
            if out_ds is None:
                raise RuntimeError("GDAL Warp failed.")
            out_ds = None  # flush
        else:
            if verbose:
                print("Translating VRT to tiled, compressed GeoTIFF (streamed)...")
            out_ds = gdal.Translate(local_filename, vrt_path, creationOptions=co)
            if out_ds is None:
                raise RuntimeError("GDAL Translate failed.")
            out_ds = None  # flush

        if verbose:
            print(f"Final mosaicked GeoTIFF saved to: {local_filename}")
        return local_filename

    except HttpError as e:
        msg = str(e)
        if "accessNotConfigured" in msg or "has not been used in project" in msg:
            raise RuntimeError(
                "Google Drive API is disabled for this service account’s project.\n"
                "Enable the **Google Drive API** in GCP → APIs & Services → Library, then retry."
            ) from e
        raise
    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise
    finally:
        # Best-effort cleanup
        try:
            shutil.rmtree(temp_dir)
            if verbose:
                print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")


In [21]:
import os
import shutil
import tempfile
import contextlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List

from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# GDAL bindings
from osgeo import gdal


def list_all_files_in_folder(
    service,
    folder_id: str,
    description: Optional[str] = None,
    *,
    include_all_drives: bool = True,
    drive_id: Optional[str] = None,
    page_size: int = 1000,
) -> List[dict]:
    """
    List ALL files in a Drive folder, with pagination.
    Filters by description substring and .tif/.TIF extension.
    Returns a list of dicts with at least {'id','name'}.
    """
    # Query: in folder, not trashed, optional name contains 'description',
    # and extension contains '.tif' or '.TIF'
    name_filter = []
    if description:
        name_filter.append(f"(name contains '{description}')")
    ext_filter = "((name contains '.tif') or (name contains '.TIF'))"
    q_parts = [f"'{folder_id}' in parents", "trashed=false", ext_filter]
    if name_filter:
        q_parts.append(" and ".join(name_filter))
    q = " and ".join(q_parts)

    kwargs = dict(
        q=q,
        pageSize=page_size,
        fields="nextPageToken, files(id, name)",
        orderBy="name",
    )
    if include_all_drives:
        kwargs.update(dict(supportsAllDrives=True, includeItemsFromAllDrives=True))
        if drive_id:
            kwargs.update(dict(corpora="drive", driveId=drive_id))
        else:
            kwargs.update(dict(corpora="allDrives"))
    else:
        kwargs.update(dict(corpora="user"))

    files, page_token = [], None
    while True:
        resp = service.files().list(pageToken=page_token, **kwargs).execute()
        files.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return files


def download_merge_from_drive(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    *,
    # Output & mosaic controls
    compress: str = "DEFLATE",          # DEFLATE|LZW|ZSTD|JPEG
    predictor: Optional[int] = None,     # 2 for integers, 3 for floats; None = let GDAL decide
    blocksize: int = 512,                # tile size for output GeoTIFF
    bigtiff: str = "YES",                # YES|IF_SAFER
    dst_srs: Optional[str] = None,       # e.g., "EPSG:5070" (if reprojection needed)
    xres: Optional[float] = None,        # set both xres & yres to enforce pixel size (e.g., 30)
    yres: Optional[float] = None,
    dst_nodata: Optional[float] = None,  # set if you need a specific nodata in output
    # Drive / listing controls
    drive_id: Optional[str] = None,      # if folder is in a Shared Drive, you can pass the Drive ID
    include_all_drives: bool = True,     # search across Shared Drives
    # Behavior controls
    check_existing: bool = True,
    n_workers: int = 4,
    verbose: bool = True,
) -> str:
    """
    Download all Drive .tif/.TIF files whose names contain `description` from `drive_folder`,
    then build a VRT and stream a mosaicked GeoTIFF to `local_filename` (no in-memory merge).
    """

    # --- Fast exit if we already have the target ---
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        if verbose:
            print(f"File already exists: {local_filename}")
        return local_filename

    # --- Temp workspace ---
    temp_dir = tempfile.mkdtemp()
    vrt_path = os.path.join(temp_dir, "mosaic.vrt")
    downloaded_paths: List[str] = []

    # --- Build Drive client ---
    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
    cred_path = os.path.abspath(service_account_file)
    credentials = service_account.Credentials.from_service_account_file(cred_path, scopes=SCOPES)
    service = build("drive", "v3", credentials=credentials)

    # Convenience flags for Shared Drives
    list_kwargs = dict(
        fields="nextPageToken, files(id, name, parents)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ) if include_all_drives else dict(fields="nextPageToken, files(id, name, parents)")

    try:
        # --- Locate the folder (paginate just in case) ---
        folder_q = (
            f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        )
        folder_kwargs = dict(
            q=folder_q,
            pageSize=1000,
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        )

        folders, page_token = [], None
        while True:
            resp = service.files().list(pageToken=page_token, **folder_kwargs).execute()
            folders.extend(resp.get("files", []))
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

        if not folders:
            raise FileNotFoundError(
                f"Folder '{drive_folder}' not found. "
                "Ensure it exists and is shared with the service account "
                f"({credentials.service_account_email}). "
                "If it is on a Shared Drive, also ensure the service account has at least Viewer."
            )
        folder_id = folders[0]["id"]

        # --- List ALL matching .tif files in that folder (paginated) ---
        files = list_all_files_in_folder(
            service,
            folder_id,
            description=description,
            include_all_drives=include_all_drives,
            drive_id=drive_id,
            page_size=1000
        )

        if not files:
            raise FileNotFoundError(
                f"No matching .tif/.TIF files found for '{description}' in '{drive_folder}'. "
                "Check filenames and sharing."
            )

        if verbose:
            print(f"Found {len(files)} file(s). Starting download to {temp_dir} ...")

        # --- Download the files (parallel if requested) ---
        if n_workers <= 1:
            for f in files:
                path = _download_file_from_drive(f["id"], f["name"], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = [
                    ex.submit(_download_file_from_drive, f["id"], f["name"], temp_dir, service_account_file)
                    for f in files
                ]
                for fut in as_completed(futures):
                    p = fut.result()
                    if p:
                        downloaded_paths.append(p)

        if not downloaded_paths:
            raise RuntimeError("Downloads finished but no files were saved.")

        if verbose:
            print(f"Building VRT from {len(downloaded_paths)} tile(s) ...")

        # --- Build VRT (lightweight, no data copy) ---
        vrt_options = {}
        if dst_nodata is not None:
            vrt_options.update(dict(srcNodata=dst_nodata, VRTNodata=dst_nodata))

        vrt = gdal.BuildVRT(vrt_path, downloaded_paths, **vrt_options)
        if vrt is None:
            raise RuntimeError("GDAL BuildVRT failed.")
        vrt = None  # flush

        # --- Stream out to GeoTIFF (no giant array in RAM) ---
        co = [
            f"BIGTIFF={bigtiff}",
            "TILED=YES",
            f"COMPRESS={compress.upper()}",
            f"BLOCKXSIZE={int(blocksize)}",
            f"BLOCKYSIZE={int(blocksize)}",
            "OVERVIEWS=AUTO", #pyramids
            "RESAMPLING=AVERAGE",
            "NUM_THREADS=ALL_CPUS"
        ]
        if predictor is not None:
            co.append(f"PREDICTOR={int(predictor)}")

        needs_warp = any(v is not None for v in (dst_srs, xres, yres, dst_nodata))
        if needs_warp:
            if verbose:
                print("Warping (streamed) to target grid...")
            warp_kwargs = dict(
                format="GTiff",
                creationOptions=co,
                multithread=True,
            )
            if dst_srs:
                warp_kwargs["dstSRS"] = dst_srs
            if xres and yres:
                warp_kwargs["xRes"] = float(xres)
                warp_kwargs["yRes"] = float(yres)
                warp_kwargs["targetAlignedPixels"] = True
            if dst_nodata is not None:
                warp_kwargs["dstNodata"] = dst_nodata

            out_ds = gdal.Warp(local_filename, vrt_path, **warp_kwargs)
            if out_ds is None:
                raise RuntimeError("GDAL Warp failed.")
            out_ds = None
        else:
            if verbose:
                print("Translating VRT to tiled, compressed GeoTIFF (streamed)...")
            out_ds = gdal.Translate(local_filename, vrt_path, creationOptions=co)
            if out_ds is None:
                raise RuntimeError("GDAL Translate failed.")
            out_ds = None

        if verbose:
            print(f"Final mosaicked GeoTIFF saved to: {local_filename}")
        return local_filename

    except HttpError as e:
        msg = str(e)
        if "accessNotConfigured" in msg or "has not been used in project" in msg:
            raise RuntimeError(
                "Google Drive API is disabled for this service account’s project.\n"
                "Enable the **Google Drive API** in GCP → APIs & Services → Library, then retry."
            ) from e
        raise
    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise
    finally:
        # Best-effort cleanup
        try:
            shutil.rmtree(temp_dir)
            if verbose:
                print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")


In [ ]:
#30 minutes
download_merge_from_drive(
    description="extendedwest_fired_bccbi",
    local_filename=os.path.join(DIR_DERIVED, "extendedwest_fired_bccbi.tif"),
    drive_folder="GEE_Exports",
    service_account_file=SERVICE_FILE,
    compress="DEFLATE",         # same as 'deflate', case-insensitive
    predictor=3,                # 3 = best for float32 data (like indices or continuous values)
    blocksize=512,              # tile size; 256–512 is good default
    bigtiff="YES",              # ensures large mosaics don’t overflow 4 GB
    #dst_srs="EPSG:5070",        # optional: force reprojection to your CONUS Albers grid
    #xres=30, yres=30,           # optional: force 30 m pixels
    #dst_nodata=-9999,           # optional: set output NoData if needed
    check_existing=True,
    n_workers=4,                # parallel downloads from Drive
    include_all_drives=True,    # safer if GEE_Exports is on a Shared Drive
    verbose=True
)

Found 210 file(s). Starting download to C:\Users\tymc5571\AppData\Local\Temp\tmpzb3fpc_j ...
Downloaded extendedwest_fired_bccbi-0000000000-0000000000.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000006656.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000013312.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000019968.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000026624.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000033280.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000039936.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000046592.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000059904.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000053248.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000066560.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000079872.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000073216.tif
Downloaded extendedwest_fired_bccbi-0000000000-0000086528.tif
Downloaded extendedwest_fired_bccbi-000

'C:\\Users\\tymc5571\\dev\\cbi-module\\data\\derived\\extendedwest_fired_bccbi.tif'